# U-Sleep for RBD-PD — Demo Notebook

This notebook runs end-to-end automatic sleep staging using U-Sleep,
fine-tuned for RBD and Parkinson's Disease populations.

**Before running:** set `filepath` in *Section 1* to point at your recording (`.set`, `.vhdr`, or `.edf`).

Two model checkpoints are available in `weights/`:

| Checkpoint | Best suited for |
|---|---|
| `Pretrained_Model.ckpt` | General healthy population |
| `Generalized_Model.ckpt` | Healthy population **and** RBD / PD individuals |

## Setup & Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from csdp_pipeline.pipeline_elements.plot_hypnogram import plotHypnoGram
from csdp_training.experiments.bids_predictor_single_file import score_file

## 1. Run Sleep Staging

Set `filepath` to your recording, choose which channels to use as EEG and EOG, then run `score_file`.

In [ ]:
# Path to the model weights (Pretrained or Generalized)
checkpoint = "weights/Pretrained_Model.ckpt"

# Path to your recording (.edf, .set, or .vhdr) — fill this in before running
filepath = ""

# EEG and EOG channel names to use (must have ≥2 each for bipolar derivations)
eeg_inputs = ["F3", "C3"]
eog_inputs = ["EOGl", "EOGr"]

labels, epochs_used, avg_outputs = score_file(
    input_path=filepath,
    eeg_inputs=eeg_inputs,
    eog_inputs=eog_inputs,
    checkpoint=checkpoint,
)

print(f"Staged {len(labels)} epochs ({len(labels) * 30 / 3600:.1f} h total)")
print(f"Valid epochs (non-NaN): {len(epochs_used)}")

## 2. Hypnogram

Plot the predicted hypnogram across the recording.

In [ ]:
# avg_outputs is shape (5, n_epochs) in model order: W=0, N1=1, N2=2, N3=3, R=4
# plotHypnoGram expects this raw model order
plot_labels = np.argmax(avg_outputs, axis=0)

# Mark NaN epochs (bad signal) as unknown (label 5)
nan_mask = np.isnan(labels)
plot_labels[nan_mask] = 5

fig, ax = plt.subplots(figsize=(14, 3))
plotHypnoGram(plot_labels, ax, title="Predicted Hypnogram")
plt.tight_layout()
plt.show()

## 3. Prediction Confidence

`avg_outputs` has shape `(5, n_epochs)` — one row per sleep stage in model order (W, N1, N2, N3, REM).
High confidence at the predicted stage indicates a clear decision; flat curves indicate ambiguity.

In [ ]:
STAGE_LABELS_MODEL = ["Wake", "N1", "N2", "N3", "REM"]  # model output order

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

plotHypnoGram(plot_labels, axes[0], title="Predicted Hypnogram")

for i, label in enumerate(STAGE_LABELS_MODEL):
    axes[1].plot(avg_outputs[i], label=label, alpha=0.75, linewidth=0.8)
axes[1].set_ylabel("Confidence")
axes[1].set_xlabel("Epoch (30 s)")
axes[1].set_ylim(0, 1)
axes[1].legend(loc="upper right", ncol=5, fontsize=8)
axes[1].set_title("Per-Stage Prediction Confidence")

plt.tight_layout()
plt.show()

## 4. Sleep Stage Distribution

Proportion of time spent in each stage — a quick summary of the staging result.

In [ ]:
# labels are in translated order: W=0, R=1, N1=2, N2=3, N3=4
STAGE_LABELS = ["Wake", "REM", "N1", "N2", "N3"]

valid_labels = labels[~np.isnan(labels)].astype(int)
counts = np.array([(valid_labels == i).sum() for i in range(5)])
pct = counts / len(valid_labels) * 100

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#d62728", "#ff7f0e", "#9467bd", "#1f77b4", "#2ca02c"]
bars = ax.bar(STAGE_LABELS, pct, color=colors)
ax.set_ylabel("Time (%)")
ax.set_title("Sleep Stage Distribution")
ax.set_ylim(0, max(pct) * 1.2)
for bar, p in zip(bars, pct):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{p:.1f}%",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()

sleep_pct = pct[1:].sum()
print(f"Sleep efficiency: {sleep_pct:.1f}%")
for label, p in zip(STAGE_LABELS, pct):
    print(f"  {label:<5}: {p:.1f}%")

## 5. Comparing Both Models

Run the Generalized model on the same recording and compare hypnograms side-by-side.
The Generalized model was additionally trained on RBD and PD recordings.

In [ ]:
checkpoint_gen = "weights/Generalized_Model.ckpt"

labels_gen, epochs_used_gen, avg_outputs_gen = score_file(
    input_path=filepath,
    eeg_inputs=eeg_inputs,
    eog_inputs=eog_inputs,
    checkpoint=checkpoint_gen,
)

plot_labels_gen = np.argmax(avg_outputs_gen, axis=0)
plot_labels_gen[np.isnan(labels_gen)] = 5

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
plotHypnoGram(plot_labels, axes[0], title="Pretrained Model")
plotHypnoGram(plot_labels_gen, axes[1], title="Generalized Model")
plt.tight_layout()
plt.show()

# Compare on valid epochs only (exclude NaN from either model)
valid_mask = ~(np.isnan(labels) | np.isnan(labels_gen))
agreement = (labels[valid_mask] == labels_gen[valid_mask]).mean() * 100
print(f"Epoch-level agreement between models: {agreement:.1f}% ({valid_mask.sum()} valid epochs)")